In [1]:
import time
import typing
import threading

from loguru import logger

In [2]:
logger.add(
    sink="../../logs/logs_01_05.json",
    level="DEBUG",
    format="{file} {thread} {function} {time} {level} {message}",
    colorize=True,
    serialize=True,
    backtrace=True,
    diagnose=True,
    enqueue=True,
    rotation="1 Mb",
    retention="1 day",
    compression="zip",
)

1

# Step 1 - Лекция "Ожидание результата от потока"

## Ожиданиe результата с помощью sleep()

In [3]:
counter: int = 0

def is_result_available() -> bool:
    # Функция для проверки наличия результата;
    # В реальном примере здесь могут быть условия для проверки результата от другого потока;
    global counter

    if counter >= 3:
        return True
    
    counter += 1
    return False

# Цикл ожиидания результата;
while not is_result_available():
    print("Ждём результата...")
    time.sleep(2)

print("Результат получен!")

Ждём результата...
Ждём результата...
Ждём результата...
Результат получен!


## Ожидание результата с помощью join()

In [4]:
def task() -> None:
    # Здесь выполняется некоторая задача
    print("Задача выполняется")

# Создание нового потока для выполнения задачи
new_thread = threading.Thread(target=task)
new_thread.start()

# Ожидание завершения нового потока
new_thread.join()
print("Задача завершена, продолжаем выполнение основного потока")

Задача выполняется
Задача завершена, продолжаем выполнение основного потока


In [5]:
def task_for_thread2() -> None:
    print("Поток 2 начал выполнение;")
    time.sleep(2)
    print("Поток 2 завершил выполнение;")

def task_for_thread1() -> None:
    print("Поток 1 начал выполнение и ожидает завершение Потока 2;")
    
    # Создание и запуск Потока 2;
    thread2: threading.Thread = threading.Thread(target=task_for_thread2)
    thread2.start()
    
    # Ожидание завершения Потока 2;
    thread2.join()
    print("Поток 1 продолжает выполнение после завершения Потока 2;")

# Создание и запуск Потока 1;
thread1: threading.Thread = threading.Thread(target=task_for_thread1)
thread1.start()
thread1.join()

Поток 1 начал выполнение и ожидает завершение Потока 2;
Поток 2 начал выполнение;
Поток 2 завершил выполнение;
Поток 1 продолжает выполнение после завершения Потока 2;


## Ожидание результата с помощью события Event()

In [6]:
# Создание объекта события;
event: threading.Event = threading.Event()

def task() -> None:
    # Задача, результат которой мы ожидаем;
    print("Задача выполняется...")
    time.sleep(5)
    # Сигнализируем о готовности результата;
    event.set()

# Создание и запуск потока;
thread: threading.Thread = threading.Thread(target=task)
thread.start()

# Ожидаем сигнала от потока;
print("Ожидаем завершения задачи...")
event.wait()
print("Задача завершена, продолжаем выполнение основного потока")

Задача выполняется...Ожидаем завершения задачи...

Задача завершена, продолжаем выполнение основного потока


## Ожидание результата с помощью методов wait() и notify()

In [7]:
condition: threading.Condition = threading.Condition()
result_ready: bool = False

def preparing_thread() -> None:
    global result_ready
    with condition:
        print("Подготовка результата...")
        time.sleep(5)
        result_ready = True
        condition.notify()

thread: threading.Thread = threading.Thread(target=preparing_thread)
thread.start()

with condition:
    while not result_ready:
        print("Ожидаем результата...")
        condition.wait()
    print("Результат получен!")

thread.join()

Подготовка результата...
Результат получен!


# Step 13

In [8]:
import threading, time
threads = []
def my_func(i):
    print(f"поток{i} начал выполняться")
    time.sleep(1)
    print(f"поток{i} завершил выполнение")

for i in range(1, 4):
    thread = threading.Thread(target=my_func, args=(i,))
    threads.append(thread)

for thread in threads:
    thread.start()

for thread in threads:
    thread.join()

print(f"{threading.main_thread().name} продолжает свое выполнение")

поток1 начал выполняться
поток2 начал выполняться
поток3 начал выполняться
поток3 завершил выполнение
поток2 завершил выполнение
поток1 завершил выполнение
MainThread продолжает свое выполнение


# Step 14

In [9]:
# Создайте целевую функцию;
def print_from_one_to_five() -> None:
    for i in range(1, 6):
        time.sleep(0.5)
        print(i)

def print_from_six_to_ten() -> None:
    for i in range(6, 11):
        time.sleep(1)
        print(i)

# Создайте и запустите потоки согласно условию задачи;
thread1: threading.Thread = threading.Thread(target=print_from_one_to_five)
thread2: threading.Thread = threading.Thread(target=print_from_six_to_ten)

thread1.start()
thread2.start()

# Дождитесь завершения потоков;
thread1.join()
thread2.join()

# Не забудьте вывести информацию о завершении работы потоков;
print("Оба потока завершили свою работу.")

1
62

3
4
7
5
8
9
10
Оба потока завершили свою работу.
